06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [21]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io
import torch
import os, os.path

# WORKING WITH 
datasetPath = 'data/'

def loadAndMergeCSV(csvRoute):
    """
    Carga de los CSV y mergeo en un único CSV. Todos los CSV estarán en la ruta 'data/', y se excluirá el CSV
    'mergedCSV.csv', producto de los mergeos si se ejecutase antes
    """
    csvRoute= 'data/'
    finalData = pd.DataFrame()
    for csvfile in [f for f in os.listdir(csvRoute) if os.path.isfile(csvRoute + f)]:
        if (csvfile != 'mergedCSV.csv' and os.path.splitext(csvfile)[1] == '.csv'):
            data = pd.read_csv(csvRoute + csvfile)
            finalData = pd.concat([data, finalData], ignore_index=True)

    return finalData

cloth_info = loadAndMergeCSV(datasetPath)

print('cloth_info shape: {}'.format(cloth_info.shape))
print('cloth_info: \n{}'.format(cloth_info))

cloth_info shape: (936, 101)
cloth_info: 
     frame         x0        y0        z0      sdf0         x1         y1  \
0        0   0.140004 -48.66119  24.86376  55.06719   0.140008 -23.659640   
1        1   0.140004 -48.66382  24.86271  55.06907   0.140008 -23.667760   
2        2   0.140004 -48.67207  24.86396  55.07700   0.140008 -23.669520   
3        3   0.140004 -48.66555  24.87239  55.07495   0.140008 -23.664530   
4        4   0.140004 -48.66825  24.85458  55.06939   0.140008 -23.673000   
..     ...        ...       ...       ...       ...        ...        ...   
931    463 -60.519120 -24.36118  18.75246  67.12456 -47.393890  -5.302965   
932    464   9.651002 -47.67019  23.54727  54.63553   5.509990 -23.601670   
933    465  60.005550 -27.82746  22.50660  70.31019  54.755860   1.202780   
934    466  42.146170 -38.25490  24.20087  62.20490  27.454250 -18.478310   
935    467 -17.700880 -46.15897  21.44164  54.26621 -19.912270 -20.980730   

           z1      sdf1         x

In [22]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=25):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        #if isinstance(csv_data, str) and "frame,x0" in csv_data:
            #self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        #else:
            #self.data = pd.read_csv(csv_data)
        self.data = csv_data
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()
        
        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'sdf'] # 'vx', 'vy', 'vz',, 'nx', 'ny', 'nz', 'md', 'u', 'v'

        self.position_prefixes = ['x', 'y', 'z']
        self.output_positions = self.data.filter(regex=r'^[xyz]\d+$')

        # Quitamos primera fila de outputs (no es el output de nada) y ultima fila de input (no tiene output)
        self.output_positions = self.output_positions.iloc[1:]
        self.data = self.data.iloc[:-1]
        

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data)
    
    def num_features(self):
        # Number of columns in a row (pos, vel, sdf, uv per vertex)
        return len(self.feature_prefixes)
    
    def num_vertex(self):
        return self.num_vertices
    
    def _get_frame_output_tensor(self, idx):
         row = self.output_positions.iloc[idx]
         frame_data = []
         for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.position_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

         tensor_data = torch.tensor(np.array(frame_data))

         return tensor_data
    
    def _get_frame_tensor(self, idx):
        row = self.data.iloc[idx]
        frame_data = []

        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

        tensor_data = torch.tensor(np.array(frame_data))

        return tensor_data
        
    def __getitem__(self, idx):
        frame_t = self._get_frame_tensor(idx)
        frame_t1 = self._get_frame_output_tensor(idx) # +1 ya no TODO: Pillar solo las columnas de pos

        return frame_t, frame_t1

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset(cloth_info)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}")
    print(batch_data)
    print(batch_frames)
    break


mean = 0
std = 0
n_samples = 0

for batch_t, _ in dataloader:
    batch_t = batch_t.float()
    
    batch_samples = batch_t.size(0)
    batch_t = batch_t.view(-1, batch_t.size(-1))  

    mean += batch_t.mean(dim=0)
    std += batch_t.std(dim=0)
    n_samples +=  batch_t.size(0)

mean /= n_samples
std /= n_samples

# evitamos division por 0
std[std < 1e-8] = 1.0

print("MEAN:", mean)
print("STD:", std)

Batch Shape: torch.Size([4, 25, 4])
tensor([[[-4.8819e+01, -3.5579e+01,  2.2116e+01,  6.4284e+01],
         [-2.6717e+01, -1.8804e+01,  4.3958e+01,  5.4553e+01],
         [-3.0088e+01, -4.3033e+01,  3.7368e+01,  6.4565e+01],
         [-3.8964e+01, -1.2237e+01,  2.2966e+01,  4.6501e+01],
         [-3.9584e+01, -3.8466e+01, -1.5157e+00,  5.5325e+01],
         [-2.1542e+01,  5.3520e+00,  4.7762e+01,  5.2008e+01],
         [-3.4447e+01, -1.3995e+01, -2.2770e+00,  3.7003e+01],
         [-1.6930e+01, -4.1507e+01, -1.2525e+01,  4.6908e+01],
         [-2.5964e+01,  9.3455e+00,  2.3541e+01,  3.5404e+01],
         [-1.1149e+01,  2.9027e+01,  4.9523e+01,  5.7441e+01],
         [-2.0122e+01, -1.8674e+01, -2.2228e+01,  3.5288e+01],
         [ 8.1944e+00, -3.7745e+01, -1.5601e+01,  4.2123e+01],
         [ 2.2452e+00, -2.1056e+01, -3.4203e+01,  4.0285e+01],
         [-8.5970e+00,  2.7901e+01,  2.4677e+01,  3.6945e+01],
         [ 1.4001e-01,  5.1330e+01,  4.9861e+01,  7.0326e+01],
         [ 1.4000e-

In [23]:
import json
# Intento de normalización uep

# Convertir tensores a listas
norm_data = {
    "mean": mean.tolist(),
    "std": std.tolist(),
    "feature_prefixes": ['x', 'y', 'z',  'sdf']# 'vx', 'vy', 'vz',, 'nx', 'ny', 'nz', 'md', 'u', 'v'
}

with open("cloth_norm_params.json", "w") as f:
    json.dump(norm_data, f)

In [ ]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Some init for my module
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Function for performing the calculation of the module.
        
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        #print(x)
        return x

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

# definir modelo, loss function y optimizer
#TODO: buscar dimensiones reales de las neuronas

# nn.Linear in PyTorch is designed to handle 3D tensors seamlessly.  
# When a 3D input tensor (e.g., batch_size, sequence_length, features) is provided,
#  the layer applies the linear transformation only to the last dimension (the features dimension),
#  preserving all other dimensions.

model = MyModule(num_inputs=4, num_hidden= 64, num_outputs=3)
# print, save, lo que sea
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(10):
    for batch_t, batch_t1 in dataloader:

        batch_t = batch_t.float()
        batch_t1 = batch_t1.float()

        batch_t = (batch_t - mean) / std

        # output (posiciones)
        mean_out = batch_t1.mean(dim=(0,1), keepdim=True)
        std_out = batch_t1.std(dim=(0,1), keepdim=True) + 1e-8
        batch_t1 = (batch_t1 - mean_out) / std_out

        pred = model(batch_t)

        loss = criterion(pred, batch_t1)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        print(f'Epoch {epoch+1}, Loss: {loss.item()}')


Epoch 1, Loss: 3374.182373046875
Epoch 1, Loss: 2531.237548828125
Epoch 1, Loss: 1955.427490234375
Epoch 1, Loss: 2369.62158203125
Epoch 1, Loss: 2516.537353515625
Epoch 1, Loss: 2320.130126953125
Epoch 1, Loss: 2791.783447265625
Epoch 1, Loss: 3061.62939453125
Epoch 1, Loss: 2043.13623046875
Epoch 1, Loss: 2132.50537109375
Epoch 1, Loss: 2484.347412109375
Epoch 1, Loss: 1669.19482421875
Epoch 1, Loss: 1585.045654296875
Epoch 1, Loss: 2358.333740234375
Epoch 1, Loss: 2973.036376953125
Epoch 1, Loss: 2292.854248046875
Epoch 1, Loss: 2842.560546875
Epoch 1, Loss: 2777.238037109375
Epoch 1, Loss: 1798.132080078125
Epoch 1, Loss: 1772.0457763671875
Epoch 1, Loss: 1604.6884765625
Epoch 1, Loss: 2793.617431640625
Epoch 1, Loss: 2957.6005859375
Epoch 1, Loss: 2665.6298828125
Epoch 1, Loss: 2611.27490234375
Epoch 1, Loss: 2507.99462890625
Epoch 1, Loss: 1633.9742431640625
Epoch 1, Loss: 1869.6700439453125
Epoch 1, Loss: 2750.06884765625
Epoch 1, Loss: 2145.881103515625
Epoch 1, Loss: 1420.1063

In [28]:
#INTENTO DE EXPORTAR A ONNX
import sys
print(sys.executable)

import onnx
import onnxruntime

print("ONNX version:", onnx.__version__)
print("ONNX Runtime version:", onnxruntime.__version__)

# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
# example_inputs = (batch_t)
example_inputs = batch_t[0:1, :, :]
onnx_program = torch.onnx.export(model, example_inputs, dynamo=True)

onnx_program.save("onnxModels/trainedModel.onnx")

E0420 11:17:57.421000 9572 site-packages\torch\export\_trace.py:1003] always_classified is unsupported.


c:\Users\Eva\miniconda3\envs\dl2024\python.exe
ONNX version: 1.21.0
ONNX Runtime version: 1.24.4
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export`... ❌
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `Sequential([...]` with Torch Script...
[torch.onnx] Obtain model graph for `Sequential([...]` with Torch Script... ❌
[torch.onnx] Obtain model graph for `Sequential([...]` with internal Dynamo apis...
[torch.onnx] Obtain model graph for `Sequential([...]` with internal Dynamo apis... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/2[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and summit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'torch._dynamo.exc.TorchRuntimeError'>: Failed running call_module fn_0(*(FakeTensor(..., size=(1, 25, 4)),), **{}):
Invalid channel dimensions

from user code:
   File "c:\Users\Eva\miniconda3\envs\dl2024\Lib\site-packages\torch\_dynamo\external_utils.py", line 40, in inner
    return fn(*args, **kwargs)

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


(Refer to the full stack trace above for more information.)

In [26]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
